# 1. Préparation de l'environnement
Nous importons les librairies nécessaires : `pandas` pour la manipulation de données, `spacy` pour le traitement linguistique, et `sklearn` pour le Machine Learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
import warnings

# Configuration de l'affichage
sns.set_style("whitegrid")
warnings.filterwarnings("ignore")

print("Librairies chargees avec succes.")

# 2. Chargement du Dataset
Le fichier contient le catalogue produits. Nous allons charger les données brutes situées dans le dossier `data/raw`.

In [ ]:
# Chargement du CSV depuis le dossier data/raw
try:
    df = pd.read_csv('../data/raw/sample-data.csv')
    print(f"Dataset charge : {df.shape[0]} produits, {df.shape[1]} colonnes.")
    display(df.head(3))
except FileNotFoundError:
    print("ERREUR : Fichier introuvable. Verifier le chemin 'data/raw/sample-data.csv'.")

# 3. Preprocessing (Nettoyage du texte)
Pour que l'algorithme comprenne nos produits, nous devons nettoyer la colonne `description`.
**Étapes :**
1. Tokenization (découpage en mots).
2. Suppression des "Stop Words" (le, la, of, the...).
3. Lemmatization (garder la racine du mot : "running" -> "run").

In [ ]:
# Chargement du modele de langue anglaise spaCy
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Modele spaCy manquant. Lancer : python -m spacy download en_core_web_sm")


def clean_text(text):
    """
    Nettoie un texte brut : passage en minuscule, lemmatisation,
    suppression des stop words et de la ponctuation.

    Parametres
    ----------
    text : str
        Description brute du produit.

    Retourne
    --------
    str
        Texte nettoye, tokens separes par des espaces.
    """
    if pd.isna(text):
        return ""

    doc = nlp(text)

    # Conservation des tokens alphabetiques, hors stop words, longueur > 2
    tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop
        and not token.is_punct
        and token.is_alpha
        and len(token) > 2
    ]

    return " ".join(tokens)


# Test rapide sur une phrase exemple
phrase_test = "The North Face creates amazing jackets for running in the mountains!"
print(f"Avant : {phrase_test}")
print(f"Apres : {clean_text(phrase_test)}")

# 3.1 Application du nettoyage
Nous appliquons cette fonction à l’ensemble du catalogue. Cela crée une nouvelle colonne `clean_description`.

In [ ]:
# Application du nettoyage a l'ensemble du catalogue
# (peut prendre 1 a 2 minutes selon la machine)
print("Nettoyage en cours...")
df['clean_description'] = df['description'].apply(clean_text)

print("Termine. Apercu :")
display(df[['description', 'clean_description']].head())

# 4. Feature Extraction : TF-IDF
Nous allons transformer nos descriptions textuelles en vecteurs numériques.
Nous utilisons **TF-IDF** (Term Frequency - Inverse Document Frequency).
* Cela donne un poids élevé aux mots rares et importants (ex: "gore-tex").
* Cela donne un poids faible aux mots trop fréquents (ex: "wear", "cloth").

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# max_features=1000 : on conserve les 1000 termes les plus discriminants
tfidf_vectorizer = TfidfVectorizer(max_features=1000)

print("Vectorisation TF-IDF en cours...")
tfidf_matrix = tfidf_vectorizer.fit_transform(df['clean_description'])

print(f"Matrice TF-IDF creee : {tfidf_matrix.shape}")
# Forme attendue : (nombre de produits, nombre de termes retenus)

# 5. Clustering : DBSCAN
Nous allons regrouper les produits similaires sans leur donner de catégories à l'avance (Apprentissage Non Supervisé).
Nous utilisons l'algorithme **DBSCAN** car il gère bien le bruit (outliers).

**Paramètres clés :**
* `metric='cosine'` : Indispensable pour du texte (mesure l'angle entre les vecteurs plutôt que la distance brute).
* `eps` : La distance maximale entre deux produits pour qu'ils soient considérés comme "voisins".
* `min_samples` : Le nombre minimum de produits pour former un groupe.

In [ ]:
from sklearn.cluster import DBSCAN

# Premier essai avec eps=0.2 (seuil de distance cosinus)
# min_samples=3 : au moins 3 produits pour former un cluster
dbscan = DBSCAN(eps=0.2, min_samples=3, metric='cosine')

print("Entrainement DBSCAN (eps=0.2)...")
clusters = dbscan.fit_predict(tfidf_matrix)

df['cluster'] = clusters

# Nombre de clusters et d'outliers
n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
n_noise = list(clusters).count(-1)

print(f"Resultat : {n_clusters} clusters trouves.")
print(f"Produits non classes (bruit, cluster -1) : {n_noise}")

# Apercu des premieres lignes triees par cluster
display(df[['description', 'cluster']].sort_values(by='cluster').head(10))

# 5.1 Optimisation des hyperparametres et visualisation
Nous augmentons `eps` pour reduire le nombre d’outliers, puis visualisons les clusters obtenus à l’aide de WordClouds.

In [ ]:
from wordcloud import WordCloud
from matplotlib.colors import LinearSegmentedColormap

# Palette aux couleurs The North Face (gris -> blanc -> rouge)
tnf_cmap = LinearSegmentedColormap.from_list(
    "tnf", ["#AAAAAA", "#FFFFFF", "#DA2427"], N=256
)

# Deuxieme iteration : eps=0.63 pour reduire le nombre d'outliers
# eps=0.2  -> 19 clusters mais 80 % outliers (trop restrictif)
# eps=0.63 -> 17 clusters et seulement 4 % outliers (bon compromis)
dbscan = DBSCAN(eps=0.63, min_samples=3, metric='cosine')
df['cluster'] = dbscan.fit_predict(tfidf_matrix)

n_clusters_final = len(set(df['cluster'])) - (
    1 if -1 in df['cluster'].values else 0
)
n_noise_final = (df['cluster'] == -1).sum()
print(f"Clusters : {n_clusters_final}")
print(f"Bruit restant : {n_noise_final} ({n_noise_final / len(df) * 100:.1f} %)")
print("\nDistribution des clusters :")
print(df['cluster'].value_counts())


def plot_wordclouds(df, n_clusters=3):
    """Affiche un WordCloud par cluster pour les n plus gros clusters."""
    unique_clusters = (
        df[df['cluster'] != -1]['cluster']
        .value_counts()
        .index[:n_clusters]
    )

    fig, axes = plt.subplots(
        1, n_clusters, figsize=(20, 5), facecolor='black'
    )
    for i, cluster_id in enumerate(unique_clusters):
        text_cluster = " ".join(
            df[df['cluster'] == cluster_id]['clean_description']
        )

        wc = WordCloud(
            background_color='black',
            max_words=50,
            colormap=tnf_cmap,
            width=800,
            height=400,
        ).generate(text_cluster)

        axes[i].imshow(wc, interpolation='bilinear')
        axes[i].set_title(
            f"Cluster {cluster_id}"
            f" ({(df['cluster'] == cluster_id).sum()} produits)",
            color='white', fontsize=14, fontweight='bold', pad=10,
        )
        axes[i].axis("off")
    plt.tight_layout()
    plt.show()


print("\nVisualisation des themes dominants :")
plot_wordclouds(df)

# 5.2 Systeme de recommandation
La fonction `find_similar_items()` retourne 5 produits appartenant au meme cluster que le produit consulte. C’est le coeur du moteur de recommandation "Vous aimerez aussi".

In [ ]:
def find_similar_items(item_id, df):
    """
    Recommande 5 produits appartenant au meme cluster que le produit donne.

    Parametres
    ----------
    item_id : int
        Index du produit dans le DataFrame.
    df : pd.DataFrame
        DataFrame contenant les colonnes 'description' et 'cluster'.

    Retourne
    --------
    pd.DataFrame ou str
        DataFrame de 5 produits similaires, ou message d'erreur.
    """
    try:
        target_cluster = df.loc[item_id, 'cluster']
    except KeyError:
        return "ID produit inconnu."

    # Les outliers (cluster -1) n'ont pas de voisins fiables
    if target_cluster == -1:
        return "Ce produit est un outlier, pas de recommandation possible."

    # Selection des produits du meme cluster, hors le produit lui-meme
    similar_items = df[
        (df['cluster'] == target_cluster)
        & (df.index != item_id)
    ]

    if len(similar_items) > 0:
        recommendations = similar_items.sample(
            n=min(5, len(similar_items))
        )
        return recommendations[['description', 'cluster']]

    return "Aucun autre produit similaire trouve dans ce cluster."


# --- Test automatique ---
sample_id = df[df['cluster'] != -1].sample(1).index[0]
print(f"Produit consulte (ID {sample_id}) :")
print(df.loc[sample_id, 'description'][:200])
print("-" * 50)
print("Vous aimerez aussi :")
display(find_similar_items(sample_id, df))

# 5.3 Test interactif
L’utilisateur peut saisir l’ID d’un produit pour obtenir des recommandations personnalisees.

In [ ]:
# --- Mode interactif (input) ---
# L'utilisateur saisit l'ID d'un produit pour obtenir des recommandations
user_input = input("Entrez l'ID d'un produit (0 a 499) : ")

try:
    user_id = int(user_input)
    print(f"\nProduit consulte (ID {user_id}) :")
    print(df.loc[user_id, 'description'][:200])
    print("-" * 50)
    print("Vous aimerez aussi :")
    result = find_similar_items(user_id, df)
    if isinstance(result, str):
        print(result)
    else:
        display(result)
except (ValueError, KeyError):
    print("ID invalide. Veuillez entrer un nombre entre 0 et 499.")

# 6. Topic Modeling : LSA (Latent Semantic Analysis)
Dernière étape : nous allons extraire les thèmes latents du corpus pour comprendre la structure globale du catalogue.
Nous utilisons **TruncatedSVD** pour réduire la dimensionnalité de la matrice TF-IDF.

In [ ]:
from sklearn.decomposition import TruncatedSVD

# 10 composantes pour extraire 10 sujets latents
n_topics = 10
lsa_model = TruncatedSVD(n_components=n_topics, random_state=42)

# Nom de variable impose par l'enonce
topic_encoded_df = lsa_model.fit_transform(tfidf_matrix)
print(f"Matrice topic : {topic_encoded_df.shape}")
print(
    f"Variance expliquee totale : "
    f"{lsa_model.explained_variance_ratio_.sum() * 100:.1f} %"
)


def display_topics(model, feature_names, no_top_words):
    """Affiche les mots-cles de chaque topic LSA."""
    print("\nThemes decouverts dans le catalogue :\n")
    for topic_idx, topic in enumerate(model.components_):
        term_indices = topic.argsort()[:-no_top_words - 1:-1]
        keywords = [feature_names[i] for i in term_indices]
        print(f"Topic {topic_idx + 1}: {', '.join(keywords)}")


feature_names = tfidf_vectorizer.get_feature_names_out()
display_topics(lsa_model, feature_names, 10)

# Extraction du topic dominant par document (+1 pour indexer a partir de 1)
df['main_topic'] = topic_encoded_df.argmax(axis=1) + 1

print("\nDistribution des topics dominants :")
print(df['main_topic'].value_counts().sort_index())

# 6.1 WordClouds des Topics LSA
Chaque topic est defini par ses mots à plus fort poids dans la composante SVD.
Contrairement au clustering, un produit peut contribuer à **plusieurs topics** simultanement.

In [ ]:
def plot_topic_wordclouds(model, feature_names, n_topics_to_show=6):
    """Genere un WordCloud par topic a partir des poids des composantes SVD."""
    n_cols = 3
    n_rows = (n_topics_to_show + n_cols - 1) // n_cols

    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(20, 5 * n_rows), facecolor='black'
    )
    axes = axes.flatten()

    for topic_idx in range(n_topics_to_show):
        topic_weights = model.components_[topic_idx]
        # On ne garde que les poids positifs pour le WordCloud
        word_weights = {
            feature_names[i]: max(topic_weights[i], 0)
            for i in range(len(feature_names))
            if topic_weights[i] > 0
        }

        if word_weights:
            wc = WordCloud(
                background_color='black',
                max_words=30,
                colormap=tnf_cmap,
                width=800,
                height=400,
            ).generate_from_frequencies(word_weights)

            axes[topic_idx].imshow(wc, interpolation='bilinear')
            axes[topic_idx].set_title(
                f"Topic {topic_idx + 1}",
                color='white', fontsize=14, fontweight='bold', pad=10,
            )
        axes[topic_idx].axis("off")
        axes[topic_idx].set_facecolor('black')

    plt.tight_layout()
    plt.show()


print("WordClouds des 6 premiers topics LSA :")
plot_topic_wordclouds(lsa_model, feature_names, n_topics_to_show=6)